# COMSOL Field Visualization (hole-centered coordinates)

Coordinate system:
- **Central hole at origin** (0, 0)
- **Neighbor hole at 150°** at (-8.70, 5.00) mm
- **y** = drift direction (hole axis)
- COMSOL rectangle: x' in [-8.70, 0], z' in [0, 5.00]
- Sector covered: 150° to 180°

Sections:
1. Geometry sketch (plan + cross-section)
2. Horizontal slices V(x', z') with fold-back to full hexagonal cell
3. Radial sections V(r, y) at angles 180° → 150°
4. E-field magnitude |E|(r, y) at the same angles
5. V(y) and Ey(y) profiles
6. E-field vector plot

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.interpolate import griddata
from matplotlib.patches import Rectangle, Circle

# ── Load and shift coordinates ──
data = np.loadtxt("../data/Mesh-MTHGEM_fields_4bar.txt", skiprows=9)
xr, yr, zr, V = data[:,0], data[:,1], data[:,2], data[:,3]

# Central hole at (x_max, z_min) → origin
x0, z0 = xr.max(), zr.min()
x = xr - x0   # x' in [-8.70, 0]
z = zr - z0   # z' in [0, 5.00]
y = yr         # drift direction

# Geometry
PITCH = 10.0
HOLE_R = 2.5
DOM_X_MIN, DOM_X_MAX = x.min(), x.max()  # [-8.70, 0]
DOM_Z_MIN, DOM_Z_MAX = z.min(), z.max()  # [0, 5.00]
xB, zB = DOM_X_MIN, DOM_Z_MAX  # neighbor hole at 150°

print(f"Central hole: (0, 0)")
print(f"150° hole:    ({xB:.2f}, {zB:.2f}), dist = {np.sqrt(xB**2+zB**2):.2f} mm")
print(f"Domain x': [{DOM_X_MIN:.2f}, {DOM_X_MAX:.2f}]")
print(f"Domain z': [{DOM_Z_MIN:.2f}, {DOM_Z_MAX:.2f}]")
print(f"y: [{y.min():.2f}, {y.max():.2f}], V: [{V.min():.0f}, {V.max():.0f}]")
print(f"Nodes: {len(x)}")

In [ ]:
# ── Helper functions ──

def interp_slab(c1, c2, val, c1g, c2g):
    """2D griddata interpolation from scattered slab data."""
    pts = np.column_stack([c1, c2])
    C1, C2 = np.meshgrid(c1g, c2g)
    q = np.column_stack([C1.ravel(), C2.ravel()])
    Vi = griddata(pts, val, q, method='linear')
    nans = np.isnan(Vi)
    if nans.any():
        Vi[nans] = griddata(pts, val, q[nans], method='nearest')
    return Vi.reshape(C1.shape)

def add_E_arrows(ax, c1g, c2g, Vmap, skip=3, color='k', scale=40):
    """Overlay normalized E-field arrows."""
    E1 = -np.gradient(Vmap, c1g[1]-c1g[0], axis=1)
    E2 = -np.gradient(Vmap, c2g[1]-c2g[0], axis=0)
    En = np.maximum(np.sqrt(E1**2 + E2**2), 1e-6)
    C1, C2 = np.meshgrid(c1g, c2g)
    s = skip
    ax.quiver(C1[::s,::s], C2[::s,::s], E1[::s,::s]/En[::s,::s], E2[::s,::s]/En[::s,::s],
              color=color, alpha=0.7, scale=scale, width=0.003, headwidth=4, headlength=5)

print("Helpers ready.")

## 1. Geometry sketch

In [ ]:
PLATE_THICK = 3.0

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 8))

# ── LEFT: Plan view ──
ax = ax1

# Hexagonal neighbors at 30° intervals starting at 30°
neighbors = []
for k in range(6):
    ang = np.radians(30 + 60*k)
    neighbors.append((PITCH*np.cos(ang), PITCH*np.sin(ang)))

# Hexagonal Voronoi cells for central + neighbors
all_holes = [(0,0)] + neighbors
for cx, cz in all_holes:
    hex_a = np.linspace(0, 2*np.pi, 7)
    hx = cx + (PITCH/np.sqrt(3)) * np.cos(hex_a)
    hz = cz + (PITCH/np.sqrt(3)) * np.sin(hex_a)
    ax.plot(hx, hz, 'b-', lw=0.6, alpha=0.3)
    c = plt.Circle((cx,cz), HOLE_R, fill=False, ec='k', lw=1, alpha=0.5)
    ax.add_patch(c)

# Highlight central and 150° holes
ax.add_patch(plt.Circle((0,0), HOLE_R, fc='lightyellow', ec='k', lw=2))
ax.add_patch(plt.Circle((xB,zB), HOLE_R, fc='lightcyan', ec='k', lw=2))
ax.text(0.3, -0.5, 'Central', fontsize=9, fontweight='bold')
ax.text(xB+0.3, zB-0.5, '150°', fontsize=9, color='teal', fontweight='bold')

# COMSOL domain rectangle
rect = Rectangle((DOM_X_MIN, DOM_Z_MIN), DOM_X_MAX-DOM_X_MIN, DOM_Z_MAX-DOM_Z_MIN,
                  fill=True, fc='red', alpha=0.08, ec='red', lw=2.5, ls='--',
                  label=f'COMSOL domain')
ax.add_patch(rect)

# Sector lines 150° and 180°
for ang_deg in [150, 180]:
    ang = np.radians(ang_deg)
    ax.plot([0, 6*np.cos(ang)], [0, 6*np.sin(ang)], 'k--', lw=0.8, alpha=0.4)
    ax.text(5.5*np.cos(ang), 5.5*np.sin(ang)+0.4, f'{ang_deg}°', fontsize=9)

# 30° arc
arc_a = np.linspace(np.radians(150), np.radians(180), 30)
ax.plot(3*np.cos(arc_a), 3*np.sin(arc_a), 'k-', lw=1)
ax.text(-2.5, 1.8, '30°', fontsize=9)

ax.set_xlim(-12, 8); ax.set_ylim(-8, 12)
ax.set_aspect('equal'); ax.legend(fontsize=9, loc='upper right')
ax.set_xlabel("x' (mm)"); ax.set_ylabel("z' (mm)")
ax.set_title("Plan view"); ax.grid(True, alpha=0.15)

# ── RIGHT: Cross-section ──
ax = ax2
ax.add_patch(Rectangle((-PITCH/2, 0), PITCH/2-HOLE_R, PLATE_THICK, fc='lightgray', ec='k', lw=1.2, label='PTFE'))
ax.add_patch(Rectangle((HOLE_R, 0), PITCH/2-HOLE_R, PLATE_THICK, fc='lightgray', ec='k', lw=1.2))
ax.add_patch(Rectangle((-HOLE_R, 0), 2*HOLE_R, PLATE_THICK, fc='lightyellow', ec='k', lw=1.5, label='Hole (gas)'))

ax.plot([-PITCH/2,-HOLE_R],[0,0],'r-',lw=4); ax.plot([HOLE_R,PITCH/2],[0,0],'r-',lw=4)
ax.plot([-HOLE_R,HOLE_R],[0,0],'r--',lw=2,alpha=0.5)
ax.text(PITCH/2+0.3, 0, 'Anode V=0', fontsize=9, color='red', va='center')

ax.plot([-PITCH/2,-HOLE_R],[PLATE_THICK]*2,'b-',lw=4)
ax.plot([HOLE_R,PITCH/2],[PLATE_THICK]*2,'b-',lw=4)
ax.text(PITCH/2+0.3, PLATE_THICK, 'Top elec.\nV~-1500V', fontsize=9, color='blue', va='center')

ax.add_patch(Rectangle((-PITCH/2, PLATE_THICK), PITCH, 10, fc='aliceblue', ec='none', alpha=0.5))
ax.text(0, PLATE_THICK+5, 'Drift\nE~105 V/cm', fontsize=11, ha='center', color='steelblue')

for yy in [0.5,1.0,1.5,2.0,2.5]:
    ax.annotate('', xy=(0,yy-0.3), xytext=(0,yy+0.3), arrowprops=dict(arrowstyle='->',color='orange',lw=1.5))
ax.text(-0.3, 1.5, 'E~4700\nV/cm', fontsize=9, ha='right', color='darkorange', fontweight='bold')

ax.plot([-PITCH/2,PITCH/2],[PLATE_THICK+10]*2,'k-',lw=3)
ax.text(PITCH/2+0.3, PLATE_THICK+10, 'Cathode\nV=-2500V', fontsize=9, va='center')

ax.set_xlim(-7,11); ax.set_ylim(-2,15)
ax.set_xlabel('r (mm)'); ax.set_ylabel('y — drift (mm)')
ax.set_title('Cross-section'); ax.legend(fontsize=9, loc='upper left')
ax.set_aspect('equal'); ax.grid(True, alpha=0.15)

fig.suptitle('MTHGEM geometry (4 bar)', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()

## 2. Horizontal sections V(x', z') with fold-back

The COMSOL domain covers 150°-180°. Using mirror symmetry at z'=0 (x-axis) and at the 150° line, we reconstruct the full hexagonal cell around the central hole.

In [ ]:
def fold_to_sector(xq, zq):
    """Fold any (x', z') point into the COMSOL sector [150°, 180°].
    
    Uses hexagonal 6-fold symmetry (60° rotations) + mirror at z'=0.
    Returns folded (xf, zf) inside the COMSOL rectangle.
    """
    xq, zq = np.asarray(xq, float), np.asarray(zq, float)
    
    # Convert to polar
    r = np.sqrt(xq**2 + zq**2)
    theta = np.arctan2(zq, xq)  # [-pi, pi]
    
    # Map theta into [0, 360) 
    theta = theta % (2 * np.pi)
    
    # Reduce to [0, 60) using 6-fold symmetry
    sector = theta % (np.pi / 3)  # [0, 60°)
    
    # Mirror: the COMSOL sector is 150°-180°, i.e. the unique 30° sector is [0°, 30°]
    # If sector > 30°, mirror: sector = 60° - sector
    sector = np.where(sector > np.pi/6, np.pi/3 - sector, sector)
    # Now sector in [0°, 30°]
    
    # Map to 150°-180°: phi_comsol = 180° - sector
    phi_comsol = np.pi - sector
    
    xf = r * np.cos(phi_comsol)
    zf = r * np.sin(phi_comsol)
    
    # Clip to domain bounds
    xf = np.clip(xf, DOM_X_MIN, DOM_X_MAX)
    zf = np.clip(zf, DOM_Z_MIN, DOM_Z_MAX)
    
    return xf, zf

# ── Horizontal slices with fold-back ──
y_cuts = [-0.2, 0.5, 1.5, 2.5, 3.5, 5.0]
y_tol = 0.4

# Full hexagonal cell grid
xg_full = np.linspace(-6, 6, 120)
zg_full = np.linspace(-6, 6, 120)
XF, ZF = np.meshgrid(xg_full, zg_full)

fig, axes = plt.subplots(2, 3, figsize=(16, 10))

for i, yc in enumerate(y_cuts):
    ax = axes.flat[i]
    mask = np.abs(y - yc) < y_tol
    print(f"y={yc:+.1f}: {mask.sum()} nodes", end="  ", flush=True)
    
    # Fold query points into COMSOL sector
    xf, zf = fold_to_sector(XF.ravel(), ZF.ravel())
    
    # Interpolate using COMSOL data at this y slab
    pts = np.column_stack([x[mask], z[mask]])
    vals = V[mask]
    query = np.column_stack([xf, zf])
    Vi = griddata(pts, vals, query, method='linear')
    nans = np.isnan(Vi)
    if nans.any():
        Vi[nans] = griddata(pts, vals, query[nans], method='nearest')
    Vi = Vi.reshape(XF.shape)
    
    # Mask outside hexagonal cell (r > pitch)
    R = np.sqrt(XF**2 + ZF**2)
    Vi[R > PITCH * 0.6] = np.nan
    
    im = ax.pcolormesh(xg_full, zg_full, Vi, cmap='RdBu_r', shading='auto', vmin=-1800, vmax=0)
    
    # Hole circle and domain rectangle
    theta = np.linspace(0, 2*np.pi, 100)
    ax.plot(HOLE_R*np.cos(theta), HOLE_R*np.sin(theta), 'k--', lw=1.2)
    ax.plot(0, 0, 'k+', ms=10)
    rect = Rectangle((DOM_X_MIN, DOM_Z_MIN), DOM_X_MAX-DOM_X_MIN, DOM_Z_MAX-DOM_Z_MIN,
                      fill=False, ec='red', lw=1.5, ls='--')
    ax.add_patch(rect)
    
    ax.set_title(f"y = {yc:.1f} mm")
    ax.set_xlabel("x' (mm)"); ax.set_ylabel("z' (mm)")
    ax.set_aspect('equal')
    fig.colorbar(im, ax=ax, shrink=0.8, label="V (V)")

print()
fig.suptitle("V(x', z') — horizontal sections with hexagonal fold-back", fontsize=14)
plt.tight_layout(); plt.show()

## 3. Radial sections V(r, y) at angles 180° → 150°

In [ ]:
phi_angles = [180, 174, 168, 162, 156, 150]

rg = np.linspace(0, 5.5, 80)
yg_r = np.linspace(-0.5, 6.0, 80)
r_tol = 0.4

fig, axes = plt.subplots(2, 3, figsize=(16, 10), sharex=True, sharey=True)

for i, phi in enumerate(phi_angles):
    ax = axes.flat[i]
    phi_rad = np.radians(phi)
    dx, dz = np.cos(phi_rad), np.sin(phi_rad)
    
    r_along = x * dx + z * dz
    r_perp  = np.abs(-x * dz + z * dx)
    mask = r_perp < r_tol
    print(f"phi={phi}°: {mask.sum()} nodes", end="  ", flush=True)
    
    if mask.sum() < 100:
        print("skip"); continue
    
    Vi = interp_slab(r_along[mask], y[mask], V[mask], rg, yg_r)
    
    im = ax.pcolormesh(rg, yg_r, Vi, cmap='RdBu_r', shading='auto', vmin=-1800, vmax=0)
    add_E_arrows(ax, rg, yg_r, Vi, skip=4, color='k', scale=40)
    ax.axvline(x=HOLE_R, color='gray', ls='--', lw=1, alpha=0.6)
    ax.axhline(y=0, color='gray', ls=':', lw=0.8)
    ax.axhline(y=3, color='gray', ls=':', lw=0.8)
    ax.set_title(f"$\\phi$ = {phi}°")
    ax.set_xlabel("r (mm)"); ax.set_ylabel("y (mm)")
    ax.set_aspect('equal')

print()
fig.colorbar(im, ax=axes, label="V (V)", shrink=0.6)
fig.suptitle("V(r, y) — radial sections through central hole, sector 180°→150°", fontsize=14)
plt.tight_layout(); plt.show()

## 4. |E| radial sections at the same angles

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 10), sharex=True, sharey=True)

for i, phi in enumerate(phi_angles):
    ax = axes.flat[i]
    phi_rad = np.radians(phi)
    dx, dz = np.cos(phi_rad), np.sin(phi_rad)
    
    r_along = x * dx + z * dz
    r_perp  = np.abs(-x * dz + z * dx)
    mask = r_perp < r_tol
    
    if mask.sum() < 100: continue
    
    Vi = interp_slab(r_along[mask], y[mask], V[mask], rg, yg_r)
    
    Er = -np.gradient(Vi, rg[1]-rg[0], axis=1)
    Ey = -np.gradient(Vi, yg_r[1]-yg_r[0], axis=0)
    Emag = np.sqrt(Er**2 + Ey**2) * 10  # V/cm
    
    im = ax.pcolormesh(rg, yg_r, Emag, cmap='inferno', shading='auto', vmin=0, vmax=8000)
    ax.axvline(x=HOLE_R, color='white', ls='--', lw=1, alpha=0.6)
    ax.axhline(y=0, color='white', ls=':', lw=0.8)
    ax.axhline(y=3, color='white', ls=':', lw=0.8)
    ax.set_title(f"$\\phi$ = {phi}°")
    ax.set_xlabel("r (mm)"); ax.set_ylabel("y (mm)")
    ax.set_aspect('equal')

fig.colorbar(im, ax=axes, label="|E| (V/cm)", shrink=0.6)
fig.suptitle("|E|(r, y) — radial sections, sector 180°→150°", fontsize=14)
plt.tight_layout(); plt.show()

## 5. V(y) and Ey(y) profiles at different radii

In [ ]:
# Profiles along y at phi=180° (along -x' axis, z'=0) at different radii
radii = [0.0, 1.0, 2.0, 2.5, 3.0, 4.0]
phi_prof = 180  # degrees
phi_rad = np.radians(phi_prof)
dx_p, dz_p = np.cos(phi_rad), np.sin(phi_rad)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

for r_val in radii:
    # Select nodes near (r_val * cos(phi), r_val * sin(phi)) in (x', z')
    xq = r_val * dx_p
    zq = r_val * dz_p
    r_from_point = np.sqrt((x - xq)**2 + (z - zq)**2)
    mask = r_from_point < 0.5
    if mask.sum() < 10:
        continue
    
    ys, Vs = y[mask], V[mask]
    # Bin
    ybins = np.linspace(-0.5, 8.0, 100)
    yc_bin = (ybins[:-1] + ybins[1:]) / 2
    V_bin = np.array([Vs[(ys>=ybins[j])&(ys<ybins[j+1])].mean()
                      if ((ys>=ybins[j])&(ys<ybins[j+1])).sum()>0 else np.nan
                      for j in range(len(ybins)-1)])
    valid = ~np.isnan(V_bin)
    ax1.plot(yc_bin[valid], V_bin[valid], '-', lw=2, label=f"r={r_val:.1f}")
    
    if valid.sum() > 5:
        Ey = -np.gradient(V_bin[valid], yc_bin[valid]) * 10
        ax2.plot(yc_bin[valid], Ey, '-', lw=2, label=f"r={r_val:.1f}")

ax1.set_xlabel("y (mm)"); ax1.set_ylabel("V (V)")
ax1.set_title(f"V(y) at $\\phi$={phi_prof}°"); ax1.legend(); ax1.grid(True, alpha=0.3)
ax1.axvline(x=0, color='gray', ls=':', lw=0.8); ax1.axvline(x=3, color='gray', ls=':', lw=0.8)

ax2.set_xlabel("y (mm)"); ax2.set_ylabel("Ey (V/cm)")
ax2.set_title(f"Ey(y) at $\\phi$={phi_prof}°"); ax2.legend(); ax2.grid(True, alpha=0.3)
ax2.axvline(x=0, color='gray', ls=':', lw=0.8); ax2.axvline(x=3, color='gray', ls=':', lw=0.8)
ax2.axhline(y=4800, color='red', ls='--', lw=0.8, alpha=0.5)
ax2.set_ylim(-500, 10000)

plt.tight_layout(); plt.show()

## 6. E-field vector plot at phi = 180°

In [ ]:
phi_vec = 180
phi_rad = np.radians(phi_vec)
dx_v, dz_v = np.cos(phi_rad), np.sin(phi_rad)

rg_v = np.linspace(0, 5.5, 30)
yg_v = np.linspace(-0.3, 6.0, 30)

r_along = x * dx_v + z * dz_v
r_perp  = np.abs(-x * dz_v + z * dx_v)
mask = r_perp < 0.5

Vi = interp_slab(r_along[mask], y[mask], V[mask], rg_v, yg_v)

dr = rg_v[1]-rg_v[0]; dy = yg_v[1]-yg_v[0]
Er = -np.gradient(Vi, dr, axis=1)
Ey = -np.gradient(Vi, dy, axis=0)
Emag = np.sqrt(Er**2 + Ey**2)
En = np.maximum(Emag, 1e-6)

RV, YV = np.meshgrid(rg_v, yg_v)

fig, ax = plt.subplots(figsize=(10, 8))
im = ax.pcolormesh(rg_v, yg_v, Emag*10, cmap='inferno', shading='auto', vmin=0, vmax=8000, alpha=0.7)
fig.colorbar(im, ax=ax, label="|E| (V/cm)")

ax.quiver(RV, YV, Er/En, Ey/En, color='white', alpha=0.8, scale=30, width=0.003, headwidth=4, headlength=5)

ax.axvline(x=HOLE_R, color='cyan', ls='--', lw=1.5, alpha=0.7)
ax.axhline(y=0, color='cyan', ls=':', lw=1); ax.axhline(y=3, color='cyan', ls=':', lw=1)
ax.text(3.0, 1.5, "PTFE\nwall", color='cyan', fontsize=10, ha='left')
ax.text(1.0, 1.5, "hole", color='cyan', fontsize=12, ha='center')
ax.text(1.0, 5.0, "drift", color='cyan', fontsize=12, ha='center')

ax.set_xlabel("r (mm)", fontsize=12); ax.set_ylabel("y — drift (mm)", fontsize=12)
ax.set_title(f"E-field at $\\phi$ = {phi_vec}°", fontsize=14)
ax.set_aspect('equal')
plt.tight_layout(); plt.show()